## Download raw files and transform them in readable CSV

In [1]:
filename = 'D_05_adz_raw_20260301'

In [ ]:
from data_download import downloader
from data_decoder import decoder

downloader(filename)
decoder(filename)

## DataFrame for the CSV (filename, timestamp)

In [2]:
import os
import pandas as pd
from pathlib import Path
from utils import date_extractor

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

path = root / f'RAW_DATA/{filename}'


rows = []
for file in os.listdir(path):
    file_path = os.path.join(path, file)

    with open(file_path, 'r') as f:
        content = f.read()
        micro_sec = int(content.split(';')[0][4:]) # in the raw data -> get the timestamp in ms

    time = date_extractor(micro_sec) # function defined in utils.py -> return the time in the day !

    date = os.path.basename(file_path).split('_')[4]
    yyyy = date[:4]
    mm = date[4:6]
    dd = date[6:]

    timestamp = f'{yyyy}-{mm}-{dd} {time}' # timestamp in format: YYYY-MM-DD HH:MM:SS
    timestamp = pd.to_datetime(timestamp)

    rows.append({
        'file': file.split('.')[0],
        'timestamp_raw': timestamp
    })

df_raw_data = pd.DataFrame(rows)
df_raw_data.head(5)

,file,timestamp_raw
0,D_05_adz_raw_20260301_0006,2026-03-01 23:57:27.667
1,D_05_adz_raw_20260301_0035,2026-03-01 00:27:14.576
2,D_05_adz_raw_20260301_0113,2026-03-01 01:03:31.185
3,D_05_adz_raw_20260301_0133,2026-03-01 01:23:46.522
4,D_05_adz_raw_20260301_0151,2026-03-01 01:42:27.866


## Get sensor data from the GeoVis

In [5]:
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv
import pandas as pd
from utils import main

env_path = 'GeoVis/DTC.env'
load_dotenv(env_path)

LOGIN = os.getenv('GEOVIS_LOGIN')
PASSWORD = os.getenv('GEOVIS_PASSWORD')

# project ID (change it for each project)
PROKECT_ID = '1113' # Ruschlikon
PROJECT_ID = '817' # Opfikon

login_data = {'Login': LOGIN, 'Password': PASSWORD}
sensor_name = '_Peak'

if PROJECT_ID == "1113":
    Projekt_DB = f"{PROJECT_ID}_0"
elif PROJECT_ID == "817":
    Projekt_DB =f"{PROJECT_ID}_7"
else:
    raise ValueError("Unknown PROJECT_ID")

datum = filename.split('_')[-1]

dfs_raw_3 = main(login_data, PROJECT_ID, Projekt_DB, sensor_name, datum)

Start Calculation: 2026-03-01T00:00:00
End Calculation  : 2026-03-02T00:00:00
Datenbezug abgeschlossen.
Gefundene Sensoren: 6
Peak Detektion abgeschlossen.
Lokale Paar- und Sensorgeschwindigkeiten gespeichert.


### Train passage over sensor

In [8]:
from GeoVis.Preprocessing.Peak_Detektion import detect_peaks

dfs_raw = detect_peaks(dfs_raw_3)

all_peaks = []
result = []

for val in dfs_raw.values():
    peaks = val["Peaks"]

    # peaks_sensor = peaks[peaks["Sensor"].str.contains("ADTC_8")] # Rueschlikon (sensor ADTC_8)
    peaks_sensor = peaks[peaks["Sensor"].str.contains("D_05")] # Opfikon (sensor D_05)

    all_peaks.append(peaks_sensor[["Time", "Value", "Sensor"]])

peaks_df = pd.concat(all_peaks, ignore_index=True)

peaks_df["Time"] = pd.to_datetime(peaks_df["Time"])
peaks_df = peaks_df.sort_values('Time').reset_index(drop=True)

time_diff = peaks_df["Time"].diff().dt.total_seconds().fillna(0)

peaks_df["train_nr"] = (time_diff>60).cumsum()+1

for train_nr, group in peaks_df.groupby("train_nr"):
    times = group["Time"].tolist()
    
    timestamp = times[0]

    train_deltas = [
        (times[i] - times[i-1]).total_seconds()
        for i in range(1, len(times))
    ]

    result.append({
        "timestamp": timestamp,
        "timestamp_serie": times,
        "delta_t": train_deltas
    })

df_geovis_time = pd.DataFrame(result)
df_geovis_time.head(5)

Peak Detektion abgeschlossen.


,timestamp,timestamp_serie,delta_t
0,2026-03-01 00:27:25.818,"[2026-03-01 00:27:25.818000, 2026-03-01 00:27:...","[0.869, 0.12, 0.51, 0.32, 0.02, 0.02, 0.16, 0...."
1,2026-03-01 01:03:40.719,"[2026-03-01 01:03:40.719000, 2026-03-01 01:03:...","[0.569, 0.13, 0.25, 0.549, 0.38, 0.05, 0.12, 0..."
2,2026-03-01 01:23:56.047,"[2026-03-01 01:23:56.047000, 2026-03-01 01:23:...","[0.529, 0.05, 0.13, 0.32, 0.669, 0.399, 0.04, ..."
3,2026-03-01 01:42:37.499,"[2026-03-01 01:42:37.499000, 2026-03-01 01:42:...","[0.749, 0.639, 0.4, 0.559, 0.18, 0.709, 0.42, ..."
4,2026-03-01 02:22:27.403,"[2026-03-01 02:22:27.403000, 2026-03-01 02:22:...","[0.449, 0.13, 0.21, 0.689, 0.42, 0.11, 0.17, 0..."


### Velocity over sensor

In [7]:
# use dfs_raw_3 -> results of the function 'main()'

rows = []

for ts, data in dfs_raw_3.items():
    sensor = data.get('geschw_pro_achse_sensor_local')

    if sensor is None:
        continue

    # replace 5 with the sensor you want to use to extract data
    if 5 in sensor:
        rows.append({
            'timestamp': ts,
            'velocity': sensor[5]
        })

df_geovis_velocity = pd.DataFrame(rows)
df_geovis_velocity.head(5)

,timestamp,velocity
0,2026-03-01 00:02:18.536,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ..."
1,2026-03-01 00:27:24.141,"[13.694444444444445, 13.694444444444445, 14.20..."
2,2026-03-01 00:31:50.177,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ..."
3,2026-03-01 01:03:39.641,"[12.544529262086513, 12.872062663185378, 12.60..."
4,2026-03-01 01:23:49.128,"[11.627358490566037, 11.627358490566037, 11.41..."


### Merge both GeoVis DataFrame and compute traing length

In [11]:
df_geovis = pd.merge_asof(
    df_geovis_time.sort_values('timestamp'),
    df_geovis_velocity.sort_values('timestamp'),
    left_on='timestamp',
    right_on='timestamp',
    direction='nearest',
    tolerance=pd.Timedelta('20s')
)

# add the length between every axis, computed using d=v*t
df_geovis['length'] = df_geovis.apply(
    lambda row: [
        a * b for a, b in zip(row["delta_t"], row["velocity"])
    ] if isinstance(row["delta_t"], list)
    and isinstance(row["velocity"], list)
    and len(row["delta_t"]) > 1
    and len(row["velocity"]) > 1
    else [],
    axis=1
)

# add the total train length, by summin the values in the length list
df_geovis['total_length'] = df_geovis['length'].apply(sum)
df_geovis = df_geovis[df_geovis['total_length'] != 0] # remove all entries where the total length is 0

df_geovis.head(5)

,timestamp,timestamp_serie,delta_t,velocity,length,total_length
0,2026-03-01 00:27:25.818,"[2026-03-01 00:27:25.818000, 2026-03-01 00:27:...","[0.869, 0.12, 0.51, 0.32, 0.02, 0.02, 0.16, 0....","[13.694444444444445, 13.694444444444445, 14.20...","[11.900472222222222, 1.6433333333333333, 7.245...",72.128289
1,2026-03-01 01:03:40.719,"[2026-03-01 01:03:40.719000, 2026-03-01 01:03:...","[0.569, 0.13, 0.25, 0.549, 0.38, 0.05, 0.12, 0...","[12.544529262086513, 12.872062663185378, 12.60...","[7.1378371501272255, 1.6733681462140992, 3.152...",146.535103
2,2026-03-01 01:23:56.047,"[2026-03-01 01:23:56.047000, 2026-03-01 01:23:...","[0.529, 0.05, 0.13, 0.32, 0.669, 0.399, 0.04, ...","[11.627358490566037, 11.627358490566037, 11.41...","[6.150872641509434, 0.5813679245283019, 1.4835...",76.416237
3,2026-03-01 01:42:37.499,"[2026-03-01 01:42:37.499000, 2026-03-01 01:42:...","[0.749, 0.639, 0.4, 0.559, 0.18, 0.709, 0.42, ...","[13.217158176943698, 13.217158176943698, 13.46...","[9.89965147453083, 8.445764075067023, 5.387978...",128.604568
4,2026-03-01 02:22:27.403,"[2026-03-01 02:22:27.403000, 2026-03-01 02:22:...","[0.449, 0.13, 0.21, 0.689, 0.42, 0.11, 0.17, 0...","[14.045584045584045, 13.656509695290858, 14.08...","[6.3064672364672365, 1.7753462603878116, 2.957...",72.758876


## Merge DataFrame from the CSV and DataFrame from GeoVIS

In [14]:
df_merged = pd.merge_asof(
    df_geovis.sort_values('timestamp'),
    df_raw_data.sort_values('timestamp_raw'),
    left_on='timestamp',
    right_on='timestamp_raw',
    direction='nearest',
    tolerance=pd.Timedelta('20s')
)

df_merged = df_merged.dropna(subset=['file', 'timestamp_raw'])

cols = ['file', 'timestamp_raw'] + [
    c for c in df_merged.columns
    if c not in ['file', 'timestamp_raw']
]

df_merged = df_merged[cols]

df_merged.head(5)

,file,timestamp_raw,timestamp,timestamp_serie,delta_t,velocity,length,total_length
0,D_05_adz_raw_20260301_0035,2026-03-01 00:27:14.576,2026-03-01 00:27:25.818,"[2026-03-01 00:27:25.818000, 2026-03-01 00:27:...","[0.869, 0.12, 0.51, 0.32, 0.02, 0.02, 0.16, 0....","[13.694444444444445, 13.694444444444445, 14.20...","[11.900472222222222, 1.6433333333333333, 7.245...",72.128289
1,D_05_adz_raw_20260301_0113,2026-03-01 01:03:31.185,2026-03-01 01:03:40.719,"[2026-03-01 01:03:40.719000, 2026-03-01 01:03:...","[0.569, 0.13, 0.25, 0.549, 0.38, 0.05, 0.12, 0...","[12.544529262086513, 12.872062663185378, 12.60...","[7.1378371501272255, 1.6733681462140992, 3.152...",146.535103
2,D_05_adz_raw_20260301_0133,2026-03-01 01:23:46.522,2026-03-01 01:23:56.047,"[2026-03-01 01:23:56.047000, 2026-03-01 01:23:...","[0.529, 0.05, 0.13, 0.32, 0.669, 0.399, 0.04, ...","[11.627358490566037, 11.627358490566037, 11.41...","[6.150872641509434, 0.5813679245283019, 1.4835...",76.416237
3,D_05_adz_raw_20260301_0151,2026-03-01 01:42:27.866,2026-03-01 01:42:37.499,"[2026-03-01 01:42:37.499000, 2026-03-01 01:42:...","[0.749, 0.639, 0.4, 0.559, 0.18, 0.709, 0.42, ...","[13.217158176943698, 13.217158176943698, 13.46...","[9.89965147453083, 8.445764075067023, 5.387978...",128.604568
4,D_05_adz_raw_20260301_0230,2026-03-01 02:22:17.540,2026-03-01 02:22:27.403,"[2026-03-01 02:22:27.403000, 2026-03-01 02:22:...","[0.449, 0.13, 0.21, 0.689, 0.42, 0.11, 0.17, 0...","[14.045584045584045, 13.656509695290858, 14.08...","[6.3064672364672365, 1.7753462603878116, 2.957...",72.758876
